This code is specifically to be run after DLT pipeline populates data in bronze layer tables for **chunk2.csv** files specifically for **city_time_series** and **zip_time_series** tables 

This script implements an Idempotent Merge Pattern designed to move data from a landing or workspace schema into the Bronze tier of a Medallion architecture. It specifically solves the problem of "Vertical Appends"—ensuring that new records are added while existing records are updated with fresh metadata, preventing duplicate entries in your Delta tables because we have chunk2.csv only for 2 csv files and we're trying to save time and have efficient compute power.

In [0]:
# --- 1. CONFIGURATION & SCHEMA RESOLUTION ---
try:
    # Attempt to load centralized configuration for environment consistency
    from schema_config import (
        source_schema,
        target_schema,
        tables
    )
except ImportError:
    # Fallback configuration for local testing or manual execution
    # city_time_series and zip_time_series are the primary Zillow datasets
    tables = ["city_time_series", "zip_time_series"]
    source_schema = "workspace.default"   # Temporary landing area
    target_schema = "data_bronze.bronze"  # Permanent Bronze tier storage

for t_name in tables:
    source_table = f"{source_schema}.{t_name}"
    target_table = f"{target_schema}.{t_name}"
    
    print(f"Processing vertical append for {t_name}...")

    # --- 2. DYNAMIC COLUMN INTERSECTION ---
    # We fetch schema metadata from both tables to avoid "Unresolved Expression" errors.
    # This ensures that if the source has extra columns not yet in Bronze, the job won't crash.
    source_cols = set(spark.read.table(source_table).columns)
    target_cols = set(spark.read.table(target_table).columns)
    
    # Identify common columns that exist in BOTH source and target
    common_cols = list(source_cols.intersection(target_cols))
    
    # Identify 'Business Keys' for the Join Condition.
    # We exclude system-generated metadata columns like 'load_dt' or '_rescued_data'
    # because they change every run and would prevent a match.
    join_cols = [c for c in common_cols if c not in ["load_dt", "source", "_rescued_data"]]
    
    # --- 3. DYNAMIC SQL FRAGMENT GENERATION ---
    
    # Build a null-safe join condition using the <=> operator.
    # This operator returns TRUE if both values are NULL, unlike the standard '='.
    join_condition = " AND ".join([f"s.{c} <=> t.{c}" for c in join_cols])
    
    # Define the columns to be used in the INSERT statement
    insert_cols = ", ".join(common_cols)
    
    # Define the values (from source 's') to be inserted
    insert_values = ", ".join([f"s.{c}" for c in common_cols])

    # --- 4. EXECUTE THE IDEMPOTENT MERGE ---
    # This SQL command atomically checks for existence before acting.
    # MATCHED: Update the metadata to show the latest ingestion attempt.
    # NOT MATCHED: Insert the entire record.
    spark.sql(f"""
        MERGE INTO {target_table} t
        USING {source_table} s
        ON {join_condition}
        WHEN MATCHED THEN
          UPDATE SET 
            t.load_dt = s.load_dt, 
            t.source = s.source
        WHEN NOT MATCHED THEN
          INSERT ({insert_cols}) 
          VALUES ({insert_values})
    """)
    
    print(f"SUCCESS: {t_name} merged into Bronze layer.")